# SRQ3 — CNN-ViT Hybrid Evaluation

Evaluates whether using the best SRQ1 CNN backbone as a spatial feature extractor with a pretrained DeiT-Small transformer encoder improves classification over the standalone CNN with a linear head.

**Architecture:**
- CNN backbone (cbam_block_post, best SRQ1 test F1) → 7×7×512 spatial feature map
- Flatten → 49 tokens of 512-dim
- Linear projection 512→384 (DeiT embedding dim)
- Learnable CLS token + positional embeddings
- Pretrained DeiT-Small transformer encoder (12 blocks, 384-dim, 6 heads)
- Linear classifier on CLS token output

**Training protocol:**
- Phase 1: freeze backbone + DeiT encoder, warm up bridge components (token_proj, cls_token, pos_embed, head)
- Phase 2: full fine-tune at reduced LR

**Backbone note:** CNN weights are loaded from arch-eval best fold. For cbam_block_post the attention module lives inside layer4 blocks and fires normally. The avgpool attention of end-placed variants is bypassed — spatial map taken from layer4 directly.

**Comparison:** Hybrid test F1/AUC/ECE vs cbam_block_post standalone (SRQ1 results).

## 1 · Paths & Imports

In [ ]:
import sys
from pathlib import Path

ABSOLUTE_PATH        = Path().resolve()
PROJECT_ROOT         = ABSOLUTE_PATH.parents[1]
DATA_DIR             = PROJECT_ROOT / "data" / "raw"
ARCH_EVAL_DIR        = PROJECT_ROOT / "data" / "experiments" / "arch-eval-results"
GRID_SEARCH_DIR      = ABSOLUTE_PATH / "grid-search-results"
HEAD_ABLATION_DIR    = ABSOLUTE_PATH / "head-ablation-results"
RESULTS_DIR          = PROJECT_ROOT / "data" / "experiments" / "cnn-vit-hybrid-results"
WEIGHTS_DIR          = RESULTS_DIR / "weights"
CURVES_DIR           = RESULTS_DIR / "training-curves"
PLOTS_DIR            = RESULTS_DIR / "plots"

for d in [RESULTS_DIR, WEIGHTS_DIR, CURVES_DIR, PLOTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

sys.path.append(str(PROJECT_ROOT))

print(f"Project root  : {PROJECT_ROOT}")
print(f"Data dir      : {DATA_DIR}")
print(f"Arch-eval dir : {ARCH_EVAL_DIR}")
print(f"Results dir   : {RESULTS_DIR}")
print(f"Weights dir   : {WEIGHTS_DIR}")


In [ ]:
import csv
import traceback
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import f1_score

import src.scripts.data      as data
import src.scripts.models    as models
import src.scripts.trainer   as trainer
import src.scripts.evaluator as evaluator
import src.scripts.utils     as utils

utils.set_seed(42)


## 2 · Data — Identical Outer Split

Same seed-42 outer 80/20 split as all other experiments. Test set is held out and not touched until Section 7.

In [ ]:
path, categories = data.get_dataset(str(DATA_DIR))
classes = data.get_classes(path, categories, visualise=False)

image_paths, labels = data.get_paths_and_labels(path, classes)
train_transform, test_transform = data.get_transforms()

X_trainval, y_trainval, X_test, y_test = data.get_trainval_test_split(
    image_paths, labels, test_split=0.20, SEED=42
)

print(f"\nTrain+Val : {len(X_trainval)}  |  Test (held out): {len(X_test)}")


## 3 · Configuration

In [ ]:
# ── Load optimal hyperparameters from grid search (reuse SRQ1 settings) ─────
OPTIMAL_CONFIG_FILE = GRID_SEARCH_DIR / "optimal_config.csv"
if not OPTIMAL_CONFIG_FILE.exists():
    raise FileNotFoundError(
        f"optimal_config.csv not found at {OPTIMAL_CONFIG_FILE}\n"
        "Run the grid search notebook first."
    )

optimal        = pd.read_csv(OPTIMAL_CONFIG_FILE).iloc[0]
LR_PHASE1      = float(optimal["lr_phase1"])
WD_PHASE1      = float(optimal["wd_phase1"])
LR_PHASE2      = float(optimal["lr_phase2"])
WEIGHT_DECAY   = float(optimal["weight_decay"])

# ── Load winning head ─────────────────────────────────────────────────────────
OPTIMAL_HEAD_FILE = HEAD_ABLATION_DIR / "optimal_head.csv"
if not OPTIMAL_HEAD_FILE.exists():
    raise FileNotFoundError(
        f"optimal_head.csv not found at {OPTIMAL_HEAD_FILE}\n"
        "Run the head ablation notebook first."
    )

WINNING_HEAD = pd.read_csv(OPTIMAL_HEAD_FILE).iloc[0]["head"]

print(f"lr_phase1    = {LR_PHASE1:.0e}")
print(f"wd_phase1    = {WD_PHASE1}")
print(f"lr_phase2    = {LR_PHASE2:.0e}")
print(f"weight_decay = {WEIGHT_DECAY:.0e}")
print(f"winning head = {WINNING_HEAD}")


In [ ]:
# ── Fixed training parameters ─────────────────────────────────────────────────
N_SPLITS    = 5
BATCH_SIZE  = 32
P1_EPOCHS   = 20
P2_EPOCHS   = 15
P1_PATIENCE = 4
P2_PATIENCE = 3
SEED        = 42

# ── Backbone — best SRQ1 test model ───────────────────────────────────────────
BACKBONE_ARCH = "cbam_block_post"

# ── Auto-select best val F1 fold weights from arch-eval for backbone init ─────
arch_results_file = ARCH_EVAL_DIR / "arch_eval_results.csv"
if not arch_results_file.exists():
    raise FileNotFoundError(
        f"arch_eval_results.csv not found at {arch_results_file}\n"
        "Run the architecture evaluation notebook first."
    )

df_arch = pd.read_csv(arch_results_file)
df_arch["error"] = df_arch["error"].fillna("")
df_backbone = (
    df_arch[(df_arch["architecture"] == BACKBONE_ARCH) & (df_arch["error"] == "")]
    .sort_values("val_f1", ascending=False)
)
if df_backbone.empty:
    raise RuntimeError(f"No completed arch-eval runs found for {BACKBONE_ARCH}")

BACKBONE_FOLD = int(df_backbone.iloc[0]["fold"])
BACKBONE_WEIGHTS = utils.weights_path_for(ARCH_EVAL_DIR / "weights", BACKBONE_ARCH, BACKBONE_FOLD)

RESULTS_FILE   = RESULTS_DIR / "hybrid_results.csv"
CSV_FIELDNAMES = [
    "backbone_arch", "backbone_fold", "head",
    "fold", "val_acc", "val_loss", "val_f1",
    "p1_epochs_run", "p2_epochs_run",
    "weights_path", "timestamp", "error"
]

total_runs = N_SPLITS
print(f"Backbone      : {BACKBONE_ARCH}  (arch-eval fold {BACKBONE_FOLD+1}, "
      f"val_f1={df_backbone.iloc[0].val_f1:.4f})")
print(f"Backbone weights: {BACKBONE_WEIGHTS}")
print(f"Head          : {WINNING_HEAD}")
print(f"Folds         : {N_SPLITS}")
print(f"Results       → {RESULTS_FILE}")
print(f"Weights       → {WEIGHTS_DIR}")


## 4 · Training Loop — 5-Fold CV

For each fold:
1. Instantiate `CNNViTHybrid` and load pre-trained backbone weights
2. Phase 1 (`bridge`) — freeze backbone + DeiT encoder, warm up token_proj, cls_token, pos_embed, head
3. Phase 2 (`bridge_and_encoder`) — keep backbone frozen, fine-tune DeiT encoder + all bridge components
4. Compute val F1, save weights, append result to CSV

The CNN backbone is frozen throughout — this isolates the question: can a transformer encoder extract more value from fixed CNN features than a linear head?

Safe to interrupt and resume — completed folds are skipped.

In [ ]:
device         = torch.device("cuda" if torch.cuda.is_available() else "cpu")
completed_runs = utils.load_completed_runs(RESULTS_FILE, [("backbone_arch", str), ("fold", int)])
run_number     = len(completed_runs)

print(f"Device : {device}")
print(f"{N_SPLITS} folds — {len(completed_runs)} already completed\n")

for fold_idx in range(N_SPLITS):

    run_key = (BACKBONE_ARCH, fold_idx)
    if run_key in completed_runs:
        print(f"SKIP  fold={fold_idx+1}/{N_SPLITS}")
        continue

    utils.set_seed(SEED)
    run_number += 1
    print(f"\n{'='*65}")
    print(f"  Run {run_number}/{total_runs}  |  fold={fold_idx+1}/{N_SPLITS}")
    print(f"{'='*65}")

    try:
        train_loader, val_loader = data.get_fold_loaders(
            X_trainval, y_trainval,
            fold_idx=fold_idx,
            train_transform=train_transform,
            test_transform=test_transform,
            n_splits=N_SPLITS,
            batch_size=BATCH_SIZE,
            SEED=SEED
        )

        # Instantiate hybrid and load pre-trained backbone weights
        model = models.get_model(
            architecture="cnn_vit_hybrid",
            backbone_arch=BACKBONE_ARCH,
            head=WINNING_HEAD
        )
        model.load_backbone_weights(str(BACKBONE_WEIGHTS), device=device)

        run_configs = {
            "phase1": {
                "num_epochs"  : P1_EPOCHS,
                "lr"          : LR_PHASE1,
                "parameters"  : "bridge",
                "optimiser"   : optim.AdamW,
                "criterion"   : nn.BCEWithLogitsLoss(),
                "weight_decay": WD_PHASE1,
            },
            "phase2": {
                "num_epochs"  : P2_EPOCHS,
                "lr"          : LR_PHASE2,
                "parameters"  : "bridge_and_encoder",
                "optimiser"   : optim.AdamW,
                "criterion"   : nn.BCEWithLogitsLoss(),
                "weight_decay": WEIGHT_DECAY,
            },
        }

        # Phase 1: bridge warm-up
        losses_p1, accs_p1, val_losses_p1, val_accs_p1 = trainer.train_model(
            model, train_loader, val_loader,
            config_name="phase1",
            train_configs=run_configs,
            early_stopping_patience=P1_PATIENCE
        )
        p1_epochs_run = len(val_accs_p1)

        utils.plot(
            losses_p1, accs_p1,
            config_name=f"phase1_fold{fold_idx}",
            val_losses=val_losses_p1, val_accuracies=val_accs_p1,
            model_name="cnn_vit_hybrid",
            save_dir=str(CURVES_DIR),
            show=False
        )

        # Phase 2: fine-tune encoder + bridge, backbone stays frozen
        losses_p2, accs_p2, val_losses_p2, val_accs_p2 = trainer.train_model(
            model, train_loader, val_loader,
            config_name="phase2",
            train_configs=run_configs,
            early_stopping_patience=P2_PATIENCE
        )
        p2_epochs_run = len(val_accs_p2)

        utils.plot(
            losses_p2, accs_p2,
            config_name=f"phase2_fold{fold_idx}",
            val_losses=val_losses_p2, val_accuracies=val_accs_p2,
            model_name="cnn_vit_hybrid",
            save_dir=str(CURVES_DIR),
            show=False
        )

        # Val F1
        model.eval()
        all_preds, all_labels_list = [], []
        with torch.no_grad():
            for imgs, lbls in val_loader:
                imgs   = imgs.to(device)
                logits = model(imgs).squeeze(1)
                preds  = (torch.sigmoid(logits) >= 0.5).long().cpu().tolist()
                all_preds.extend(preds)
                all_labels_list.extend(lbls.tolist())
        val_f1 = f1_score(all_labels_list, all_preds, average="binary", zero_division=0)

        # Save weights
        wpath = utils.weights_path_for(WEIGHTS_DIR, BACKBONE_ARCH, fold_idx)
        utils.save_weights(model, wpath)

        result_row = {
            "backbone_arch"  : BACKBONE_ARCH,
            "backbone_fold"  : BACKBONE_FOLD,
            "head"           : WINNING_HEAD,
            "fold"           : fold_idx,
            "val_acc"        : round(float(val_accs_p2[-1]),   6),
            "val_loss"       : round(float(val_losses_p2[-1]), 6),
            "val_f1"         : round(float(val_f1),            6),
            "p1_epochs_run"  : p1_epochs_run,
            "p2_epochs_run"  : p2_epochs_run,
            "weights_path"   : str(wpath),
            "timestamp"      : datetime.now().isoformat(timespec="seconds"),
            "error"          : "",
        }
        utils.append_result(RESULTS_FILE, CSV_FIELDNAMES, result_row)
        completed_runs.add(run_key)

        print(f"  val_f1={val_f1:.4f}  val_acc={val_accs_p2[-1]:.4f}  "
              f"val_loss={val_losses_p2[-1]:.4f}  "
              f"(p1={p1_epochs_run}ep  p2={p2_epochs_run}ep)")
        print(f"  Weights → {wpath}")

    except Exception as e:
        error_msg = f"{type(e).__name__}: {str(e)}"
        print(f"ERROR -- {error_msg}")
        traceback.print_exc()
        error_row = {
            "backbone_arch": BACKBONE_ARCH, "backbone_fold": BACKBONE_FOLD,
            "head": WINNING_HEAD, "fold": fold_idx,
            "val_acc": "", "val_loss": "", "val_f1": "",
            "p1_epochs_run": "", "p2_epochs_run": "",
            "weights_path": "",
            "timestamp": datetime.now().isoformat(timespec="seconds"),
            "error": error_msg,
        }
        utils.append_result(RESULTS_FILE, CSV_FIELDNAMES, error_row)

print(f"\n{'='*65}")
print("HYBRID TRAINING COMPLETE")
print(f"Results → {RESULTS_FILE}")
print(f"Weights → {WEIGHTS_DIR}")


## 5 · Results Analysis

CV summary — for training context only. Definitive results are in Section 7 (test set).

In [ ]:
df_raw = pd.read_csv(RESULTS_FILE)
df_raw["error"] = df_raw["error"].fillna("")
df_ok   = df_raw[df_raw["error"] == ""].copy()
df_fail = df_raw[df_raw["error"] != ""].copy()

for col in ["val_f1", "val_acc", "val_loss"]:
    df_ok[col] = df_ok[col].astype(float)

print(f"Successful runs : {len(df_ok)} / {total_runs}")
print(f"Failed runs     : {len(df_fail)}")
if len(df_fail) > 0:
    print(df_fail[["fold", "error"]].to_string(index=False))


In [ ]:
if not df_ok.empty:
    print("CNN-ViT Hybrid — CV Summary\n")
    print(f"  mean val F1   : {df_ok.val_f1.mean():.4f} ± {df_ok.val_f1.std():.4f}")
    print(f"  mean val acc  : {df_ok.val_acc.mean():.4f} ± {df_ok.val_acc.std():.4f}")
    print(f"  mean val loss : {df_ok.val_loss.mean():.4f} ± {df_ok.val_loss.std():.4f}")
    print()
    print("Per-fold breakdown:")
    print(f"  {'Fold':<6} {'val_f1':>8} {'val_acc':>8} {'val_loss':>10} {'p1_ep':>6} {'p2_ep':>6}")
    print(f"  {'-'*50}")
    for _, r in df_ok.sort_values("fold").iterrows():
        print(f"  {int(r.fold)+1:<6} {r.val_f1:>8.4f} {r.val_acc:>8.4f} "
              f"{r.val_loss:>10.4f} {int(r.p1_epochs_run):>6} {int(r.p2_epochs_run):>6}")

    # Comparison against standalone backbone CV F1
    df_arch = pd.read_csv(ARCH_EVAL_DIR / "arch_eval_results.csv")
    df_arch["error"] = df_arch["error"].fillna("")
    backbone_cv_f1 = (
        df_arch[(df_arch["architecture"] == BACKBONE_ARCH) & (df_arch["error"] == "")]
        ["val_f1"].mean()
    )
    hybrid_cv_f1 = df_ok.val_f1.mean()
    print(f"\n  Hybrid CV mean F1    : {hybrid_cv_f1:.4f}")
    print(f"  {BACKBONE_ARCH} CV mean F1 : {backbone_cv_f1:.4f}")
    print(f"  Δ (hybrid − backbone): {hybrid_cv_f1 - backbone_cv_f1:+.4f}")


## 6 · Final Test Set Evaluation — SRQ3

Auto-selects the best val F1 fold, loads its weights, evaluates once on the held-out test set. Compares against cbam_block_post standalone results from SRQ1.

In [ ]:
# ── Auto-select best val F1 fold ─────────────────────────────────────────────
if df_ok.empty:
    raise RuntimeError("No completed folds — run Section 5 first.")

best_row  = df_ok.sort_values("val_f1", ascending=False).iloc[0]
BEST_FOLD = int(best_row["fold"])

print(f"Selected fold {BEST_FOLD+1} (val_f1={best_row.val_f1:.4f}  "
      f"val_loss={best_row.val_loss:.4f})")


In [ ]:
device      = torch.device("cuda" if torch.cuda.is_available() else "cpu")
test_loader = data.get_test_loader(X_test, y_test, test_transform, BATCH_SIZE)
print(f"Device: {device}")


In [ ]:
wpath = utils.weights_path_for(WEIGHTS_DIR, BACKBONE_ARCH, BEST_FOLD)

utils.set_seed(SEED)
model = models.get_model(
    architecture="cnn_vit_hybrid",
    backbone_arch=BACKBONE_ARCH,
    head=WINNING_HEAD
)
model = utils.load_weights(model, wpath, device=device)

print(f"\n{'='*60}")
print(f"  CNN-ViT Hybrid  (backbone={BACKBONE_ARCH}  fold {BEST_FOLD+1})")
print(f"{'='*60}")

acc, prec, rec, f1, auc, ece, conf, report = evaluator.evaluate_model(
    model=model, test_loader=test_loader, device=device
)

# Load SRQ1 standalone baseline for comparison
arch_test_file = ARCH_EVAL_DIR / "arch_eval_test_results.csv"
if arch_test_file.exists():
    df_srq1 = pd.read_csv(arch_test_file, index_col="architecture")
    if BACKBONE_ARCH in df_srq1.index:
        standalone_f1  = float(df_srq1.loc[BACKBONE_ARCH, "test_f1"])
        standalone_auc = float(df_srq1.loc[BACKBONE_ARCH, "test_auc"])
        standalone_ece = float(df_srq1.loc[BACKBONE_ARCH, "test_ece"])
        print(f"\nSRQ3 Comparison:")
        print(f"  {'':30} {'F1':>8} {'AUC':>8} {'ECE':>8}")
        print(f"  {'-'*54}")
        print(f"  {'CNN-ViT Hybrid':<30} {f1:>8.4f} {auc:>8.4f} {ece:>8.4f}")
        print(f"  {BACKBONE_ARCH+' (standalone)':<30} {standalone_f1:>8.4f} "
              f"{standalone_auc:>8.4f} {standalone_ece:>8.4f}")
        print(f"  {'Δ (hybrid − standalone)':<30} {f1-standalone_f1:>+8.4f} "
              f"{auc-standalone_auc:>+8.4f} {ece-standalone_ece:>+8.4f}")


In [ ]:
TEST_RESULTS_FILE = RESULTS_DIR / "hybrid_test_results.csv"

test_row = {
    "backbone_arch"   : BACKBONE_ARCH,
    "selected_fold"   : BEST_FOLD,
    "test_f1"         : round(f1,   4),
    "test_auc"        : round(auc,  4),
    "test_ece"        : round(ece,  4),
    "test_acc"        : round(acc,  4),
    "test_prec"       : round(prec, 4),
    "test_rec"        : round(rec,  4),
    "conf_matrix"     : conf.tolist(),
}

pd.DataFrame([test_row]).to_csv(TEST_RESULTS_FILE, index=False)
print(f"Test results saved → {TEST_RESULTS_FILE}")


In [ ]:
# ── SRQ3 comparison bar chart ─────────────────────────────────────────────────
if arch_test_file.exists() and BACKBONE_ARCH in df_srq1.index:

    metrics      = ["F1", "AUC-ROC", "ECE"]
    hybrid_vals  = [f1,  auc,  ece]
    standalone_vals = [standalone_f1, standalone_auc, standalone_ece]

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    fig.suptitle("SRQ3 — CNN-ViT Hybrid vs Standalone CNN (Test Set)",
                 fontsize=13, fontweight="bold")

    # Panel 1: F1, AUC, ECE grouped bars
    ax  = axes[0]
    x   = np.arange(len(metrics))
    w   = 0.35
    ax.bar(x - w/2, standalone_vals, w, label=f"{BACKBONE_ARCH} (standalone)",
           color="#888", alpha=0.80)
    ax.bar(x + w/2, hybrid_vals,     w, label="CNN-ViT Hybrid",
           color="#4C72B0", alpha=0.85)
    ax.set_xticks(x)
    ax.set_xticklabels(metrics, fontsize=11)
    ax.set_ylabel("Score", fontsize=11)
    ax.set_title("Metric comparison", fontsize=11, fontweight="bold")
    ax.legend(fontsize=9)
    ax.spines[["top", "right"]].set_visible(False)
    for xi, (sv, hv) in enumerate(zip(standalone_vals, hybrid_vals)):
        ax.text(xi - w/2, sv + 0.002, f"{sv:.4f}", ha="center", fontsize=8)
        ax.text(xi + w/2, hv + 0.002, f"{hv:.4f}", ha="center", fontsize=8)

    # Panel 2: Δ bar per metric
    ax2     = axes[1]
    deltas  = [f1-standalone_f1, auc-standalone_auc, ece-standalone_ece]
    colours = ["#4C72B0" if d >= 0 else "#E05C2A" for d in deltas]
    # For ECE lower is better — flip sign for visualisation
    ece_delta_display = -(ece - standalone_ece)
    deltas_display    = [f1-standalone_f1, auc-standalone_auc, ece_delta_display]
    colours_display   = ["#4C72B0" if d >= 0 else "#E05C2A" for d in deltas_display]
    ax2.bar(x, deltas_display, width=0.5, color=colours_display, alpha=0.85)
    for xi, d in enumerate(deltas_display):
        ax2.text(xi, d + 0.001 * (1 if d >= 0 else -1),
                 f"{d:+.4f}", ha="center",
                 va="bottom" if d >= 0 else "top", fontsize=9)
    ax2.axhline(0, color="#555", linewidth=0.9, linestyle="--")
    ax2.set_xticks(x)
    ax2.set_xticklabels(["ΔF1", "ΔAUC", "ΔECE\n(+ve = better)"], fontsize=10)
    ax2.set_ylabel("Δ (hybrid − standalone)", fontsize=11)
    ax2.set_title("Gain from hybrid", fontsize=11, fontweight="bold")
    ax2.spines[["top", "right"]].set_visible(False)

    plt.tight_layout()
    utils.save_fig(fig, PLOTS_DIR, "srq3_test_results", formats=("svg",))
    plt.show()
